# 10.02 · Observability in depth: LangSmith and OpenTelemetry

**Module 10 · Testing, Evaluation & Observability** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 2–3 h |
| 📋 Prerequisites | Lesson 10.01; Lesson 01.08 (first traces, `mastery.evals.meta`). |
| 🔑 Keys | none (optional: `LANGSMITH_API_KEY` for the 🌐 cells) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-15 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Trace** your own functions with `@traceable`, and inspect the run tree locally without sending anything.
2. **Record** prompt version, model ID and git commit as metadata on every trace, and group turns into **threads**.
3. **Mask** secrets and personal data before a trace leaves your process, and know what the built-in rules miss.
4. **Export** the same work as OpenTelemetry spans, with GenAI attribute names, and route them to LangSmith or any OTLP backend.
5. **Query** traces with the SDK, filtered by metadata, to feed the next error analysis.

## 📚 Official reading (verified 2026-10-15)

| Page | Read it for |
|---|---|
| [LangSmith Observability](https://docs.langchain.com/langsmith/observability) | Traces, runs, projects: the overview |
| [Custom instrumentation](https://docs.langchain.com/langsmith/annotate-code) | `@traceable`, `trace` context manager, `run_type` |
| [Add metadata and tags](https://docs.langchain.com/langsmith/add-metadata-tags) | `metadata=` and `tags=` on traces, at definition or call time |
| [Configure threads](https://docs.langchain.com/langsmith/threads) | The `session_id` / `thread_id` metadata keys, on **all** runs of a thread |
| [Cost tracking](https://docs.langchain.com/langsmith/cost-tracking) | How token counts become costs per trace |
| [Redact secrets](https://docs.langchain.com/langsmith/redact-secrets) · [Prevent logging of sensitive data](https://docs.langchain.com/langsmith/mask-inputs-outputs) | `create_secret_anonymizer`, `create_anonymizer`, `hide_inputs`, `LANGSMITH_HIDE_INPUTS` |
| [Query traces with the SDK](https://docs.langchain.com/langsmith/export-traces) | `list_runs` with filters |
| [Trace with OpenTelemetry](https://docs.langchain.com/langsmith/trace-with-opentelemetry) · [Route between destinations](https://docs.langchain.com/langsmith/trace-replicas) | `LANGSMITH_OTEL_ENABLED`, `LANGSMITH_OTEL_ONLY`, the OTLP endpoint, Collector fan-out |
| [OpenTelemetry Python](https://opentelemetry.io/docs/languages/python/) · [GenAI semantic conventions](https://github.com/open-telemetry/semantic-conventions-genai) | Tracers, spans, exporters; the `gen_ai.*` attribute names (status: Development) |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "langsmith[otel]>=0.14.7,<1" "opentelemetry-sdk>=1.42.1,<2" "opentelemetry-exporter-otlp>=1.42.1,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langsmith", "opentelemetry-sdk", "opentelemetry-exporter-otlp-proto-http",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/10-testing-evaluation-observability/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm10-observability'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m10-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

In [ ]:
# 🧪 Offline: the Orbita Cloud desk from Project 09 (handoffs pattern, rule-based offline model).
from langgraph.checkpoint.memory import InMemorySaver

from mastery.agents.support import SupportDesk, build_handoffs_desk, make_support_tools, offline_model
from mastery.agents.support_traces import generate_inputs, reference_label, reference_note, run_traces

OUT = REPO_ROOT / "data" / "outputs" / "m10"
OUT.mkdir(parents=True, exist_ok=True)
desk = SupportDesk.load()
app = build_handoffs_desk(offline_model, make_support_tools(desk), checkpointer=InMemorySaver())
inputs = generate_inputs(60, seed=10)
traces = run_traces(app, inputs, desk)
print(len(traces), "traces;", {k: sum(t["kind"] == k for t in traces) for k in ("realistic", "edge", "adversarial")})

**You should now see** `60 traces; {'realistic': 34, 'edge': 14, 'adversarial': 12}`: the same trace set as Lesson 10.01.

---
## Part 1 · Traces, runs, threads

| Term | Meaning | On the desk |
|---|---|---|
| **Run** | One unit of work: a model call, a tool call, a function | `billing` calling `lookup_invoices` |
| **Trace** | A tree of runs for one request, rooted at the top-level run | One customer message, end to end |
| **Thread** | The traces of one conversation, linked by a `thread_id` (or `session_id`) metadata key | Two turns of the same customer |
| **Project** | Where traces land (`LANGSMITH_PROJECT`, here `m10-observability`) | One per environment or experiment |
| **Metadata / tags** | Key-value facts and labels on a run, used for filtering | `prompt_version`, `git_sha`, `m10` |

LangChain and LangGraph trace themselves when tracing is on. Your own code (the function that wraps a turn, a parser, a policy check) traces with `@traceable`.

---
## Part 2 · `@traceable`, inspected locally

`tracing_context(enabled="local")` builds the run tree in memory and sends **nothing**, so you can see exactly what LangSmith would receive. We wrap one desk turn and record what the next error analysis needs: the prompt version (a content hash of the triage prompt), the model, the git commit and the package versions (`mastery.evals.run_metadata`, Lesson 01.08).

In [ ]:
# 🧪 Offline: one desk turn as a traced function; the run tree stays in this process.
from langsmith import traceable, tracing_context
from langsmith.run_helpers import get_current_run_tree

from mastery.agents.support import TRIAGE_PROMPT
from mastery.evals import run_metadata

META = run_metadata(prompt_version="triage-v1", prompt=TRIAGE_PROMPT, model="offline-support-model")
captured = []

@traceable(name="desk_turn", run_type="chain", tags=["m10", "handoffs"])
def desk_turn(message: str, thread_id: str) -> dict:
    captured.append(get_current_run_tree())
    result = app.invoke({"messages": [{"role": "user", "content": message}]}, {"configurable": {"thread_id": thread_id}})
    return {"answer": result["messages"][-1].content}

with tracing_context(enabled="local", metadata={**META, "thread_id": "thread-ana-1"}):
    desk_turn("Hi, I'm C-1003. Please refund INV-3001 ($19), I was charged by mistake.", "thread-ana-1")
root = captured[0]
print(root.name, root.run_type, root.tags)
print({k: root.metadata[k] for k in ("prompt_version", "prompt_hash", "model", "git_sha", "thread_id")})
print("outputs:", root.outputs)

**You should now see** `desk_turn chain ['m10', 'handoffs']`, a metadata dict with `prompt_version: 'triage-v1'`, a 12-character `prompt_hash`, the model, your current `git_sha` (or `None` outside a git clone) and `thread_id`, and the refund answer.

### What just happened?

The trace now **says which version produced it**. When Lesson 10.01's error analysis runs next month, you can slice failures by `prompt_version` or `git_sha` instead of guessing which deploy changed things. `prompt_hash` changes whenever the prompt text changes, even if someone forgets to bump `prompt_version`.

---
## Part 3 · Threads: every run of a conversation carries the same id

LangSmith groups traces into a thread by the metadata key `thread_id` (or `session_id`). The docs insist on setting it on **all** runs, child runs included, or filtering and token counts per thread break. Setting it once in `tracing_context(metadata=...)` does that: children inherit the parent's metadata.

In [ ]:
# 🧪 Offline: two turns, one thread. Every run in both trees carries the same thread_id.
captured.clear()
with tracing_context(enabled="local", metadata={**META, "thread_id": "thread-ben-7"}):
    desk_turn("I'm customer C-1002, can you list my invoices?", "thread-ben-7")
    desk_turn("Thanks. Is station sync down right now?", "thread-ben-7")

def walk(run, depth=0):
    yield depth, run
    for child in run.child_runs:
        yield from walk(child, depth + 1)

for root in captured:
    runs = list(walk(root))
    print(root.outputs["answer"][:60], "| runs:", len(runs), "| thread ids:", {r.metadata.get("thread_id") for _, r in runs})

**You should now see** two lines, each with `runs: 20` and `thread ids: {'thread-ben-7'}`: the graph's nodes, model calls and tool calls appear as children of `desk_turn`, and every one inherited the thread id from the context.

> 💡 **Per-agent views:** in a multi-agent trace (Module 09), LangChain tags each model run with `lc_agent_name` in its metadata. Filter on it in the LangSmith UI to see only the `billing` agent's calls across hundreds of traces.

---
## Part 4 · Mask before it leaves your process

A trace holds whatever your code saw: customer IDs, emails, sometimes API keys pasted into a chat. Masking has to happen **in the client**, before upload. LangSmith gives you three levels:

| Tool | What it does |
|---|---|
| `create_secret_anonymizer()` | A curated rule set for common credential formats; matches become `[SECRET_DETECTED]` |
| `create_anonymizer([...rules])` | Your own regex rules (customer IDs, emails, phone numbers) |
| `hide_inputs=` / `hide_outputs=` on `Client`, or `LANGSMITH_HIDE_INPUTS=true` | Replace whole payloads (for zero-retention requirements) |

Test your rules on realistic payloads before trusting them:

In [ ]:
# 🧪 Offline: what the built-in secret rules catch, and what they miss. Fake keys, built at runtime.
from langsmith.anonymizer import create_anonymizer, create_secret_anonymizer

fake_openai = "sk-proj-" + "FAKE" * 10
fake_groq = "gsk_" + "FAKE" * 12
payload = {"message": f"I'm C-1003 (chen@li.example). My keys: {fake_openai} and {fake_groq}"}

secrets_only = create_secret_anonymizer()
print("built-in :", secrets_only(payload)["message"])

ours = create_anonymizer([
    {"pattern": r"gsk_[A-Za-z0-9]{20,}", "replace": "[GROQ_KEY]"},
    {"pattern": r"[\w.+-]+@[\w-]+\.[\w.]+", "replace": "<email>"},
    {"pattern": r"\bC-\d{4}\b", "replace": "<customer>"},
])
print("ours     :", ours(secrets_only(payload))["message"])

**You should now see** the OpenAI-style key replaced by `[SECRET_DETECTED]` while the **Groq key stays in clear text** (checked with langsmith 0.14.9), then our rules masking the Groq key, the email and the customer ID.

### What just happened?

A curated rule set is a starting point, not a guarantee: it didn't know the provider this course uses. Add rules for every credential and identifier your system handles, and plug both into the client: `Client(anonymizer=...)` (one anonymizer; compose them in a function like the last line does).

> ⚠️ **Production note: masking is not access control.**
> The docs say it plainly: a redacted trace still holds the prompts and tool results it was built from. Mask in the client with rules you've tested on real payloads (include your providers' key formats), restrict who can read each tracing project, and set retention to the shortest period that still covers your error analysis.

---
## Part 5 · OpenTelemetry: the same story, vendor-neutral

OpenTelemetry (OTel) is the open standard for traces: a **tracer** creates **spans** (a span ≈ a run), a **span processor** batches them, an **exporter** sends them over **OTLP** to any backend (LangSmith, Jaeger, Datadog, Grafana...). A **Collector** in between can fan the same spans out to several backends.

For model calls and agents, the **GenAI semantic conventions** name the attributes: `gen_ai.operation.name`, `gen_ai.request.model`, `gen_ai.usage.input_tokens`, `gen_ai.tool.name`... They live in their own repository and are marked **Development** (checked 2026-10-15), so names can still change.

Below, a private tracer provider with an in-memory exporter shows exactly what would be exported, with no backend:

In [ ]:
# 🧪 Offline: one desk turn as OTel spans with GenAI attribute names, kept in memory.
from mastery.evals.otel import GENAI, in_memory_tracing, span_table

provider, exporter = in_memory_tracing("orbita-support-desk")
tracer = provider.get_tracer("m10.desk")
trace = traces[0]

with tracer.start_as_current_span("invoke_agent desk") as turn:
    turn.set_attribute(GENAI["operation"], "invoke_agent")
    turn.set_attribute("thread.id", "thread-ana-1")
    turn.set_attribute("app.prompt_version", META["prompt_version"])
    for tool in trace["tools"]:
        with tracer.start_as_current_span(f"execute_tool {tool}") as span:
            span.set_attribute(GENAI["operation"], "execute_tool")
            span.set_attribute(GENAI["tool"], tool)
rows = span_table(exporter.get_finished_spans(), [GENAI["operation"], GENAI["tool"]])
table(rows, ["name", "parent", GENAI["operation"], GENAI["tool"]])

**You should now see** a root span `invoke_agent desk` and one `execute_tool ...` child per tool the first trace used, each with its `gen_ai.*` attributes. (Durations are in the `ms` column of `rows`; they're tiny here.)

### What just happened?

Same structure as a LangSmith trace (a root with children), but in a format every observability vendor reads. That's the portable skill: instrument once with OTel, choose (or change) backends with configuration.

---
## Part 6 · Sending OTel to LangSmith (and elsewhere)

The docs describe three setups (verified 2026-10-15):

1. **LangChain/LangGraph app → OTel:** set `LANGSMITH_OTEL_ENABLED=true` (with `LANGSMITH_TRACING=true` and your key). LangChain runs are exported as OTel spans. Add `LANGSMITH_OTEL_ONLY=true` to send **only** OTel (no native LangSmith export), for example when a Collector routes everything.
2. **Any code → LangSmith's OTLP endpoint:** point a standard OTLP/HTTP exporter at LangSmith with two variables (below). `langsmith.integrations.otel.configure(project_name=...)` wires this up for supported instrumentors.
3. **Fan-out:** an OpenTelemetry Collector receives your spans once and exports them to LangSmith *and* another backend (`traces_endpoint: https://api.smith.langchain.com/otel/v1/traces`).

The SDK's `Client` also takes `tracing_mode=` (`"otel"`, or `"hybrid"` for both at once); the docs page teaches the environment variables, so prefer those.

In [ ]:
# 🧪 Offline: the two OTLP variables for LangSmith, and the classic 404 caught before it happens.
from mastery.evals.otel import check_otlp_endpoint, langsmith_otlp_env

for key, value in langsmith_otlp_env(os.environ["LANGSMITH_PROJECT"]).items():
    print(f"{key}={value}")
print()
for candidate in ("https://api.smith.langchain.com/otel", "https://api.smith.langchain.com/otel/v1/traces"):
    print(f"{candidate:<50} ->", check_otlp_endpoint(candidate) or "ok")

**You should now see** `OTEL_EXPORTER_OTLP_ENDPOINT=https://api.smith.langchain.com/otel` and a headers line with a `<LANGSMITH_API_KEY>` placeholder and `Langsmith-Project=m10-observability`, then `ok` for the base URL and a warning for the one ending in `/v1/traces`: OTLP/HTTP exporters append that path themselves, so it would be doubled and return 404.

> ⚠️ **Production note: OTel spans can be dropped silently.**
> The docs warn that spans whose parent never arrives (orphaned children) are not shown, and a wrong endpoint fails quietly in a background thread. After wiring OTel, send one test trace and check it appears with all its children before you rely on the pipeline; alert when trace volume drops to zero.

> 🌐 Talks to LangSmith (free Developer plan is enough; no model tokens). Without `LANGSMITH_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
# 🌐 LangSmith: send one traced turn for real, then find it again by metadata.
if HAVE_LANGSMITH_KEY:
    from langsmith import Client

    with tracing_context(enabled=True, metadata={**META, "thread_id": "thread-live-1"}):
        desk_turn("I'm C-1003 and my account is locked, I can't log in.", "thread-live-1")
    client = Client()
    client.flush()
    found = list(client.list_runs(project_name=os.environ["LANGSMITH_PROJECT"], is_root=True, limit=5,
                                  filter='and(eq(metadata_key, "prompt_version"), eq(metadata_value, "triage-v1"))'))
    print(len(found), "root run(s) with prompt_version=triage-v1 (indexing can take a few seconds)")
else:
    print("⏭️ Skipped: LANGSMITH_API_KEY is not set (see the setup cell).")

**You should now see** `⏭️ Skipped` without a key. With one, `1` or more runs (re-run the cell if it says 0: ingestion takes a moment). Open the project in LangSmith and filter by `prompt_version` to see the same thing in the UI.

### Cost tracking, in one paragraph

LangSmith computes cost per run from the token counts a model integration reports and a price table per model; for models it doesn't know you set prices yourself. Your Module 01 habit still applies: `mastery.pricing` gives the course's per-model prices, so you can check the dashboard's numbers. A trace with no token counts has no cost, which is a common reason for "free" runs in a report.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Thread view shows only the first message, or token totals per thread are wrong | `thread_id` set on the root run only | Put it in `tracing_context(metadata=...)` (children inherit) or on every run |
| `LangSmithMissingAPIKeyWarning` in a notebook | A `Client` was created without a key (fine offline) | Ignore offline; set `LANGSMITH_API_KEY` for 🌐 cells |
| OTLP exporter returns 404 | `OTEL_EXPORTER_OTLP_ENDPOINT` ends in `/v1/traces` | Use the base `.../otel`; the full path goes only in a per-signal `endpoint=` |
| Spans missing in LangSmith | Orphaned children, wrong headers, or the process exited before export | `provider.shutdown()` / `force_flush()` at exit; check one test trace end to end |
| A key or email shows up in a trace | The anonymizer's rules didn't cover it | Add a rule, test it on a real payload (Part 4), re-check after every new provider |
| Can't tell which deploy caused a regression | No version metadata on traces | `run_metadata(...)` on every trace: prompt version and hash, model, git SHA |

## ⚠️ Production notes

> ⚠️ **Production note: masking is not access control.** (Part 4)

> ⚠️ **Production note: OTel spans can be dropped silently.** (Part 6)

> ⚠️ **Production note: tracing costs money and has retention tiers.**
> Free Developer plans include a monthly trace allowance (5k base traces per month, checked 2026-10-15); base retention is 14 days, extended is 180 days (the SaaS maximum since 2026-09-14), and some features upgrade traces to extended automatically (Lesson 10.06). Sample high-volume traffic, trace everything in staging, and keep the version metadata so a sample is still sliceable.

## 🏋️ Exercises

1. **(Easy)** Add `customer_tier` (`"Starter"`, `"Pro"`, `"Team"`) to the metadata of Part 2's trace for the right customer. Why is a tier safer to log than the customer ID?
2. **(Medium)** Write a `hide_inputs` function that keeps the message but masks customer IDs and emails, and check it with `Client(hide_inputs=...)` on Part 4's payload (call the function directly; no key needed).
3. **(Medium)** Extend Part 5: add a `chat` span per model call with `gen_ai.request.model` and made-up `gen_ai.usage.input_tokens`, then sum tokens per trace from the exported spans.
4. **(Stretch, 🌐)** With a key, set `LANGSMITH_OTEL_ENABLED=true` and `LANGSMITH_TRACING=true` in a fresh kernel, run one desk turn, and compare the trace with Part 6's native one. What looks different?

<details>
<summary><b>Show solutions</b></summary>

**1.** `{**META, "thread_id": ..., "customer_tier": desk.customers["C-1003"]["plan"]}`. A tier slices failures by segment without identifying anyone; an ID links every trace to a person (personal data, retention and deletion rules apply).

**2.**
```python
import re
def hide(inputs: dict) -> dict:
    text = str(inputs)
    text = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", "<email>", text)
    return {"masked": re.sub(r"\bC-\d{4}\b", "<customer>", text)}
print(hide(payload))
# Client(hide_inputs=hide, hide_outputs=hide) applies it to every run before upload.
```

**3.**
```python
with tracer.start_as_current_span("invoke_agent desk"):
    for model_call in range(2):
        with tracer.start_as_current_span("chat gpt-oss-20b") as span:
            span.set_attribute(GENAI["model"], "openai/gpt-oss-20b")
            span.set_attribute(GENAI["input_tokens"], 400)
spans = exporter.get_finished_spans()
print(sum(s.attributes.get(GENAI["input_tokens"], 0) for s in spans))
```

**4.** The OTel version shows the same tree, but attributes follow OTel naming, and anything your code adds as span attributes arrives as metadata. Some LangSmith-native fields (feedback links, rich run types) depend on the mapping; check the docs' attribute mapping table before switching a production project.
</details>

## 🔎 Docs drill (5 min)

You set `OTEL_EXPORTER_OTLP_ENDPOINT=https://api.smith.langchain.com/otel/v1/traces` and every export fails with 404. Why, and when **is** the `/v1/traces` path correct?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Trace with OpenTelemetry](https://docs.langchain.com/langsmith/trace-with-opentelemetry) and find the note under the endpoint variables.
2. `OTEL_EXPORTER_OTLP_ENDPOINT` is a **base** URL: OTLP/HTTP exporters append `/v1/traces` themselves, so the path is doubled.
3. The full path is right where an exporter or Collector takes a per-signal endpoint (`endpoint=` in code, `traces_endpoint:` in the Collector YAML): the page's fan-out example uses it there.

</details>

## ✅ Checkpoint

- [ ] I traced a function with `@traceable` and inspected its run tree locally.
- [ ] Every trace carries prompt version, prompt hash, model and git SHA; turns share a `thread_id`.
- [ ] I tested masking rules on a realistic payload and found what the built-in rules miss.
- [ ] I exported spans with GenAI attribute names and know the LangSmith OTLP settings (and the 404 trap).
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Check the [GenAI semantic conventions](https://github.com/open-telemetry/semantic-conventions-genai) and log in [`docs/research-log.md`](../../docs/research-log.md): their status (2026-10-15: Development) and whether any attribute used here (`gen_ai.operation.name`, `gen_ai.tool.name`, `gen_ai.request.model`, `gen_ai.usage.input_tokens`) was renamed. Also note whether `create_secret_anonymizer` now recognises Groq keys.

## Summary

- Runs form traces; traces form threads (`thread_id` on every run); projects hold them.
- `@traceable` + `run_metadata` make every trace say which prompt, model and commit produced it.
- Mask in the client with tested rules; masking is not access control.
- OpenTelemetry carries the same structure to any backend; GenAI conventions name the attributes (still evolving).
- `list_runs` with metadata filters feeds the next error analysis.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 10-03-datasets-and-experiments.ipynb** → [`10-03-datasets-and-experiments.ipynb`](10-03-datasets-and-experiments.ipynb).

[Module 10 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)